In [11]:
import os
import re
import math
import torch
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from gidd.eval.generation_info import GenerationInfoHandler
from datasets import load_from_disk

mask_token_id = 9

In [ ]:
# Load data

base_dir = "generation_info"
combination_row_ids = [1, 2]

dir_path_dict = {}
meta_dict = {}
history_dict = {}
logits_dict = {}
confidence_t_0_dict = {}
marginals_dict = {}
change_events_table_dict = {}
forward_calls_dict = {}
prune_correct_dict = {}
beam_search_forward_calls_dict = {}
beam_search_branch_correctness_dict = {}
beam_search_beam_correctness_dict = {}
denoise_config_dict = {}
combination_rows = []

for combination_row in combination_row_ids:
    dir_path = f"{base_dir}/combination_{combination_row}"
    meta, history, logits, confidence_t_0, marginals, change_events_table, forward_calls, prune_correct, beam_search_forward_calls, beam_search_branch_correctness, beam_search_beam_correctness = GenerationInfoHandler.load_all(dir_path)
    dir_path_dict[combination_row] = dir_path
    meta_dict[combination_row] = meta
    history_dict[combination_row] = history
    logits_dict[combination_row] = logits
    confidence_t_0_dict[combination_row] = confidence_t_0
    marginals_dict[combination_row] = marginals
    change_events_table_dict[combination_row] = change_events_table
    forward_calls_dict[combination_row] = forward_calls
    prune_correct_dict[combination_row] = prune_correct
    beam_search_forward_calls_dict[combination_row] = beam_search_forward_calls
    beam_search_branch_correctness_dict[combination_row] = beam_search_branch_correctness
    beam_search_beam_correctness_dict[combination_row] = beam_search_beam_correctness
    denoise_config_dict[combination_row] = meta['sampling_strategy']['p_denoise']

    sampling_parameters = meta['sampling_strategy'].copy()
    if isinstance(sampling_parameters.get('p_denoise', None), dict):
        for k, v in sampling_parameters['p_denoise'].items():
            sampling_parameters[f'p_denoise_{k}'] = v
        del sampling_parameters['p_denoise']
    sampling_parameters['combination_row'] = combination_row
    combination_rows.append(sampling_parameters)

combinations_df = pd.DataFrame(combination_rows)

print(meta_dict[combination_row_ids[0]])

{'accuracy': 0.6625, 'correctly_filled_cells': 0.8523920440673828, 'nfe': 1.25, 'time_taken_s': 302.6657693386078, 'speed_samples_per_s': 21.14543714006849, 'created_at': '2025-11-16 14:18:22 GMT', 'num_samples': 6400, 'max_seq_len': 81, 'collect_history': True, 'collect_logits': True, 'collect_confidence_t_0': True, 'collect_marginals': True, 'collect_change_events': True, 'collect_forward_calls': True, 'collect_prune_correct': False, 'collect_beam_search_forward_calls': False, 'collect_beam_search_branch_correctness': False, 'general_sampling': {'device': 0, 'seed': 1, 'checkpoint': 'checkpoints/gidd_0_2/300_epochs', 'dataset': 'hard', 'num_samples': 6400, 'num_denoising_steps': 80, 'num_self_correction_steps': 0, 'batch_size': 64, 'min_p': 0.0, 'compile_torch': 0, 'combinations_row': 1}, 'sampling_strategy': {'strategy': 'gidd_change_low_confidence_positions', 'time_steps': 'inferred', 'score_mask_position': 'MDM_max', 'score_position_for_change': None, 'select_position': None, 'sel